In [ ]:
import os
from datetime import datetime, timedelta
from typing import Optional

import pandas as pd
from sqlmodel import SQLModel, Field, Session, create_engine, select


In [ ]:
%run set_secrets.ipynb

In [ ]:
if (os.environ.get('TINKOFF_API_TOKEN', '') == ''):
    print('Environment variable TINKOFF_API_TOKEN not setted!!')
else:
    TOKEN = os.environ["TINKOFF_API_TOKEN"]

if (os.environ.get('DB_USER', '') == ''):
    print('Environment variable DB_USER not setted!!')
else:
    DB_USER= os.environ["DB_USER"]

if (os.environ.get('DB_PASSWORD', '') == ''):
    print('Environment variable DB_PASSWORD not setted!!')
else:
    DB_PASSWORD = os.environ["DB_PASSWORD"]

if (os.environ.get('DB_NAME', '') == ''):
    print('Environment variable DB_NAME not setted!!')
else:
    DB_NAME = os.environ["DB_NAME"]

if (os.environ.get('DB_HOST', '') == ''):
    print('Environment variable DB_HOST not setted!!')
else:
    DB_HOST = os.environ["DB_HOST"]

if (os.environ.get('DB_PORT', '') == ''):
    print('Environment variable DB_PORT not setted!!')
else:
    DB_PORT = os.environ["DB_PORT"]



In [ ]:
# Определение модели данных
class StockData(SQLModel, table=True):
    time: datetime = Field(primary_key=True)
    open: float
    high: float
    low: float
    close: float
    volume: int

In [ ]:
DATABASE_URL =  f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(DATABASE_URL)
SQLModel.metadata.create_all(engine)

In [ ]:

# Функция для сохранения данных в базу данных
def save_to_db(df: pd.DataFrame):
    try:
        with Session(engine) as session:
            for index, row in df.iterrows():
                stock_data = StockData(
                    time=row['time'],
                    open=row['open'],
                    high=row['high'],
                    low=row['low'],
                    close=row['close'],
                    volume=row['volume']
                )
                session.add(stock_data)
            session.commit()
    except Exception as e:
        print(f"Error save_data: {e}")
        return None

In [ ]:
def get_data(depth_days: int) -> Optional[pd.DataFrame]:
    try:
        date = datetime.now() - timedelta(days=depth_days)
        
        with Session(engine) as session:
            statement = select(StockData).where(StockData.time > date).order_by(StockData.time)
            result = session.exec(statement).all()
            
            if result:
                df = pd.DataFrame([row.dict() for row in result])
                return df
            else:
                return None
    except Exception as e:
        print(f"Error get_data: {e}")
        return None

In [ ]:
engine = create_engine(DATABASE_URL)
SQLModel.metadata.create_all(engine)

In [ ]:
from datetime import datetime
from dateutil import parser
# Вспомогательная функция для экспорта данных из CSV в базу данных
def csv_to_postgres(csv_file_path: str, db_url: str):
	engine = create_engine(db_url)
	SQLModel.metadata.create_all(engine)
	df = pd.read_csv(csv_file_path)
	stock_data_list = []

	for index, row in df.iterrows():
		try:
			stock_data_list.append(
			StockData(
				time=parser.isoparse(row['time']),  # Используем isoparse для обработки временных меток с часовым поясом
				open=row['open'],
				high=row['high'],
				low=row['low'],
				close=row['close'],
				volume=row['volume']
				)
			)
		except ValueError as e:
			print(f"Ошибка преобразования данных в строке {index + 1}: {e}")

	# Записываем данные в базу данных
	with Session(engine) as session:
		session.add_all(stock_data_list)
		session.commit()

In [ ]:
#csv_to_postgres('sber_data.csv', DATABASE_URL)

In [ ]:

def create_test_data():
    return pd.DataFrame({
        "time": [datetime.now() - timedelta(days=i) for i in range(5)],
        "open": [100 + i for i in range(5)],
        "high": [105 + i for i in range(5)],
        "low": [95 + i for i in range(5)],
        "close": [102 + i for i in range(5)],
        "volume": [1000 + i * 100 for i in range(5)],
    })

# Функция для тестирования save_to_db и get_data
def test_save_and_get_data():
	# Создаем тестовые данные
	test_data = create_test_data()

	# Сохраняем данные в базу
	save_to_db(test_data)

	# Получаем данные из базы
	retrieved_data = get_data(depth_days=10)

	# Проверяем, что данные корректно извлечены
	assert retrieved_data is not None, "Данные не были извлечены из базы"
	assert len(retrieved_data) == len(test_data), "Количество строк не совпадает"

    # Проверяем, что извлеченные данные совпадают с исходными
	for col in test_data.columns:
		if col != "time":
			assert col in retrieved_data.columns, f"Колонка {col} отсутствует в извлеченных данных"
			assert all(retrieved_data[col] == test_data[col]), f"Данные в колонке {col} не совпадают"

	print("Тест пройден успешно!")




In [ ]:
#test_save_and_get_data()